# Title

In [ ]:
import os
from pathlib import Path

import pandas as pd
from sklearn.metrics import roc_auc_score

from bank_marketing.config import load_config
from bank_marketing.data import load_raw
from bank_marketing.eda import (
    numeric_correlations,
    quality_summary,
    target_rate_by_category,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT) # config paths are relative to the repository root
config = load_config("configs/default.yaml")
df = load_raw(config.data.raw_path)
y = (df["y"] == "yes").astype(int)
df.shape

In [ ]:
summary = quality_summary(df)
pd.Series(
    {
        "rows": summary["n_rows"],
        "positive_rate": round(summary["positive_rate"], 4),
        "exact_duplicates": summary["exact_duplicate_rows"],
        "pdays_999_share": round(summary["pdays_sentinel_share"], 4),
    }
)

In [ ]:
pd.Series(summary["unknown_level_counts"]).sort_values(ascending=False)

In [ ]:
rates = target_rate_by_category(df)
rates[rates["feature"].isin(["poutcome", "contact", "month", "default"])]

In [ ]:
print(df.groupby("y")["duration"].describe().round(1))
print("ROC-AUC of duration alone:", round(roc_auc_score(y, df["duration"]), 3))

In [ ]:
pd.crosstab(df["pdays"] == 999, df["poutcome"], margins=True)

In [ ]:
econ = ["emp_var_rate", "cons_price_idx", "cons_conf_idx", "euribor3m", "nr_employed"]
numeric_correlations(df).loc[econ, econ]

In [ ]:
block_rate = y.groupby(df.index // 2000).mean().round(3)
block_rate.to_frame("positive_rate").T

# Title

- P1
- P2
- [docs/eda_findings.md]